# Lab 7BB: Matrix Factorization on Million Song Dataset 

### AIML ZG527: Audio Analysis
**Authored by:** Seetha Parameswaran, BITS Pilani WILP
### Learning Objectives
- Build a user-song interaction matrix from implicit play-count data
- Train SVD-based matrix factorization for latent taste factors
- Generate top-k song recommendations for a sample user
---
### Setup and Import Libraries
```
!pip install "cython<3.0" --break-system-packages
!pip install implicit --break-system-packages 
```
---
### Dataset: Million Song Dataset — Taste Profile (Echo Nest)
Implicit feedback: (user, song, play_count) triplets.

```
!wget http://millionsongdataset.com/sites/default/files/challenge/train_triplets.txt.zip
!unzip train_triplets.txt.zip
!shuf -n 10000 train_triplets.txt > train_triplets_10k.txt
```

In [16]:
import pandas as pd, numpy as np
from scipy.sparse import csr_matrix
import implicit


## Load triplets, convert play counts to implicit rating scale

In [ ]:
import random

with open('/home/seetha/PythonScriptsCourses/PythonScriptsCourses/Audio Analysis/Data/milion_song_train_triplets.txt') as f:
    lines = f.readlines()

random.seed(42)
sample = random.sample(lines, 10000)

with open('/home/seetha/PythonScriptsCourses/PythonScriptsCourses/Audio Analysis/Data/million_song_train_triplets_10k.txt', 'w') as f:
    f.writelines(sample)

TRIPLETS_PATH = '/home/seetha/PythonScriptsCourses/PythonScriptsCourses/Audio Analysis/Data/million_song_train_triplets_10k.txt'

In [17]:
df = pd.read_csv(TRIPLETS_PATH, sep='\t', names=['user_id','song_id','play_count'])
print(df.shape)
df['conf'] = 1 + np.log1p(df['play_count'])  # confidence weighting for implicit feedback

(0, 3)


## Build sparse user-item matrix and train ALS (matrix factorization)

In [15]:
users = df['user_id'].astype('category')
songs = df['song_id'].astype('category')
user_idx, song_idx = users.cat.codes, songs.cat.codes

user_song_matrix = csr_matrix((df['conf'], (user_idx, song_idx)))

algo = implicit.als.AlternatingLeastSquares(factors=20, iterations=20, regularization=0.02)
algo.fit(user_song_matrix)

Exception ignored on calling ctypes callback function: <function _ThreadpoolInfo._find_modules_with_dl_iterate_phdr.<locals>.match_module_callback at 0x7fa548b0d8b0>
Traceback (most recent call last):
  File "/home/seetha/PythonScriptsCourses/PythonScriptsCourses/lib/python3.8/site-packages/threadpoolctl.py", line 400, in match_module_callback
    self._make_module_from_path(filepath)
  File "/home/seetha/PythonScriptsCourses/PythonScriptsCourses/lib/python3.8/site-packages/threadpoolctl.py", line 515, in _make_module_from_path
    module = module_class(filepath, prefix, user_api, internal_api)
  File "/home/seetha/PythonScriptsCourses/PythonScriptsCourses/lib/python3.8/site-packages/threadpoolctl.py", line 606, in __init__
    self.version = self.get_version()
  File "/home/seetha/PythonScriptsCourses/PythonScriptsCourses/lib/python3.8/site-packages/threadpoolctl.py", line 646, in get_version
    config = get_config().split()
AttributeError: 'NoneType' object has no attribute 'split'


  0%|          | 0/20 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Evaluate: RMSE + top-k recommendations for a sample user 

In [ ]:
sample_user_idx = 0
song_ids, scores = algo.recommend(sample_user_idx, user_song_matrix[sample_user_idx], N=10)
recommended_songs = songs.cat.categories[song_ids]
print('Top 10 recommendations:', list(zip(recommended_songs, scores)))

## Summary

In this lab, you learned:

1. **Implicit Feedback**: Play counts converted to log-scaled ratings
2. **Matrix Factorization**: SVD learns latent user/song factors ($R \approx U \times V^T$)
3. **Evaluation**: RMSE on held-out interactions
4. **Recommendation**: Top-k unseen songs ranked by predicted preference

### Key Takeaways
- Latent factors capture hidden taste dimensions without hand-crafted features
- Implicit feedback (plays) is noisier than explicit ratings — log-scaling helps stabilize training
- `surprise`'s SVD is CPU-friendly; `implicit`'s ALS scales better for larger catalogs